# Task 1.1 — Core Contribution / Architecture
**Paper:** Efficient Additive Kernels via Explicit Feature Maps  
**Authors:** Andrea Vedaldi, Andrew Zisserman  
**Venue:** IEEE TPAMI, Vol. 34(3), 2012  
**Student:** Nikhil Raj | Roll No: 230080

---

## Step-by-Step Method Description

---

### Step 1: Input Representation
- **What happens:** The method takes as input a D-dimensional histogram vector **x** ∈ ℝ₊ᴰ, where every entry is non-negative. These are feature histograms commonly used in computer vision — for example, bag-of-visual-words vectors or HOG descriptors.
- **Reference:** Section 2, Equation (7) — the paper explicitly states that the data domain is the non-negative semiaxis ℝ₊ᴰ, described as the set of all D-dimensional histograms.
- **Purpose:** Establishes the domain assumption. All downstream operations (logarithm, square root) are only valid for non-negative inputs. This is not a coincidence — additive kernels like χ² and the intersection kernel are specifically designed to measure similarity between histograms.

---

### Step 2: Selection of an Additive Homogeneous Kernel
- **What happens:** The user selects one kernel from the additive homogeneous family. The paper provides closed-form support for four kernels: Hellinger's k(x,y)=√(xy), chi-squared k(x,y)=2xy/(x+y), intersection k(x,y)=min(x,y), and Jensen-Shannon. The selected kernel is applied additively across all D dimensions: K(**x**,**y**) = Σᵢ k(xᵢ,yᵢ).
- **Reference:** Section 2.1, Figure 1 (top row) — the table lists each kernel's formula, its signature K(λ), its spectrum κ(ω), and its feature map Ψ_ω(x).
- **Purpose:** Additive decomposition (Eq. 7) is the structural property that makes the entire method possible. Because the kernel sums independently over dimensions, the feature map can be applied to each dimension separately — this is what makes the transformation O(D) rather than O(D²).

---

### Step 3: Compute the Kernel Signature K(λ)
- **What happens:** Each homogeneous kernel kₕ(x,y) can be rewritten in terms of a scalar function called the *signature*: K(λ) = kₕ(e^(λ/2), e^(-λ/2)), where λ = log(y/x). The signature fully characterises the kernel in log-space and reduces the problem of analysing a 2D kernel to analysing a 1D function.
- **Reference:** Section 2, Equations (2) and (3). The signature for χ² is K(λ) = sech(λ/2), for intersection it is K(λ) = e^(-|λ|/2) — both listed in Figure 1.
- **Purpose:** The signature is the bridge between the kernel and its Fourier analysis. Without this reduction to a scalar function, the Bochner's theorem argument in the next step would not be applicable in 1D.

---

### Step 4: Derive the Exact Infinite-Dimensional Feature Map via Bochner's Theorem
- **What happens:** Bochner's theorem (Eq. 9) states that any positive definite function K(λ) is the Fourier transform of a non-negative measure with density κ(ω). Applying this to the kernel signature yields the spectrum κ(ω), and from it the exact (infinite-dimensional) feature map for a single scalar input x:  
  Ψ_ω(x) = e^(iω log x) √(x^γ κ(ω))  
  This map satisfies k(x,y) = ⟨Ψ(x), Ψ(y)⟩ exactly.
- **Reference:** Section 3, Equation (12) for γ-homogeneous kernels. The spectrum κ(ω) for χ² is sech(πω), listed in Figure 1.
- **Purpose:** This is the theoretical backbone. It proves that an exact feature map *exists* in infinite dimensions. The next step makes it finite and usable.

---

### Step 5: Construct the Finite Approximate Feature Map (Homogeneous Kernel Map)
- **What happens:** The infinite feature map is approximated by periodising the signature with period Λ and sampling the resulting discrete spectrum at integer harmonics j = 0, ±1, ..., ±n. This yields the finite real-valued feature map for each scalar component xᵢ:  

  Ψ̂_j(xᵢ) = { √(xᵢ^γ κ̂₀),                          j = 0  
               { √(2xᵢ^γ κ̂_{(j+1)/2}) cos(j+1)/2 · L · log(xᵢ)),  j > 0, odd  
               { √(2xᵢ^γ κ̂_{j/2}) sin(j/2 · L · log(xᵢ)),          j > 0, even  

  For sample_steps n=2 (i.e. j = 0,1,2,3,4), each scalar xᵢ maps to **5 values** (2n+1 = 5).
- **Reference:** Section 4.2, Equation (19). The parameter Λ is selected automatically using the curve Λ*(n) = a√n + b + c from Section 5.1 and Figure 3.
- **Purpose:** This is the core contribution of the paper. It makes the kernel map finite, deterministic (no random sampling), data-independent (no training data needed to compute), and computable in closed form for all common additive kernels.

---

### Step 6: Concatenate Component-wise Maps to Form φ(x)
- **What happens:** Because the kernel is additive, the full D-dimensional feature map is obtained by independently applying Ψ̂ to each of the D components of x and stacking the results:  
  φ(**x**) = [ Ψ̂(x₁) | Ψ̂(x₂) | ... | Ψ̂(xD) ] ∈ ℝ^{(2n+1)·D}  
  For D=64, n=2, this gives a 320-dimensional vector.
- **Reference:** Section 3, Equation (13) — the additive combination uses the direct sum (⊕) of per-component maps, yielding dimensionality nD.
- **Purpose:** This step transforms the entire input vector in O((2n+1)·D) time — linear in D. This is the efficiency gain. No kernel matrix is ever computed.

---

### Step 7: Train a Linear SVM on the Transformed Features
- **What happens:** The transformed dataset {φ(**x**₁), ..., φ(**x**ₙ)} is passed to a standard linear SVM solver (the paper uses LIBLINEAR). The SVM learns a weight vector **w** ∈ ℝ^{(2n+1)·D} and bias b.
- **Reference:** Section 8 (Experiments) — all approximated feature map experiments use LIBLINEAR; the exact kernel experiments use LIBSVM. Table 1 reports accuracy and training time comparisons.
- **Purpose:** The whole point is that once φ(**x**) is computed, the problem is a plain linear classification problem. Any linear solver works — SGD, LIBLINEAR, PEGASOS — without any modification.

---

### Step 8: Inference — Single Dot Product
- **What happens:** For a new test point **x***, compute φ(**x***) using the same fixed map from Step 5–6, then evaluate the linear decision function: f(**x***) = **w** · φ(**x***) + b. The predicted class is ŷ = sign(f(**x***)).
- **Reference:** Section 1, Introduction — "A linear SVM is given by the inner product F(x) = ⟨w, x⟩ between a data vector x and a weight vector w."
- **Purpose:** Inference cost is O((2n+1)·D) — a fixed constant independent of the number of training samples. Compare this to the exact kernel SVM where inference costs O(n_sv · D), which grows with the training set size.

---

## Final Summary Sentence

This paper solves the problem of expensive nonlinear SVM training and inference by introducing the *homogeneous kernel map* — a data-independent, closed-form, finite-dimensional feature map derived from 1D Fourier analysis of kernel signatures — which transforms histogram data into a (2n+1)×D dimensional representation that approximates additive homogeneous kernels (χ², intersection, Hellinger's) with exponentially decaying error, enabling standard linear SVM solvers to match the accuracy of exact nonlinear kernel SVMs while being orders of magnitude faster in both training and testing.
